# Stronger Filtered Back-Translation Experiment

Project: **Improving English-Bangla Translation in Low-Resource Settings Using Back-Translation**

This notebook runs one stronger final test after the valid Colab result where back-translation reduced BLEU. It keeps the same deterministic 5,000/500/500 train/validation/test split, increases synthetic candidates, filters synthetic pairs, and retrains the improved model.

No results are prefilled. The notebook writes measured outputs only to `/content/final_stronger_bt_experiment`.

## Runtime

Use **Runtime -> Change runtime type -> T4 GPU**.

The notebook uses fp32 (`FP16=False`) because the previous fp16 mT5-small run produced NaN losses and empty predictions. If Colab runs out of memory, lower `BATCH_SIZE` from 4 to 2 and increase gradient accumulation.

In [ ]:
!pip -q install -U "transformers>=4.41,<5" "datasets>=2.19,<4" "sentencepiece>=0.2" "sacrebleu>=2.4" "accelerate>=0.30" "protobuf>=4" "pandas" "matplotlib" 

In [ ]:
import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import sacrebleu
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found. Set Colab runtime to T4 GPU.")

SEED = 42
set_seed(SEED)
random.seed(SEED)

MODEL_NAME = "google/mt5-small"
DATASET_NAME = "ai4bharat/samanantar"
DATASET_CONFIG = "bn"
DATASET_PARQUET = "hf://datasets/ai4bharat/samanantar/bn/train-00000-of-00005.parquet"

# Keep same 5k/500/500 split as the previous valid Colab run.
TRAIN_SIZE = 5000
VAL_SIZE = 500
TEST_SIZE = 500
BASE_POOL_SIZE = 7500

# Stronger BT settings.
SYNTHETIC_CANDIDATES = 3000
SYNTHETIC_TARGET = 2000
MIN_FILTERED_SYNTHETIC = 1500

BASELINE_EPOCHS_IF_NEEDED = 3
REVERSE_EPOCHS = 3
IMPROVED_EPOCHS = 3

BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
GENERATION_MAX_NEW_TOKENS = 96
GENERATION_BATCH_SIZE = 16
BLEU_TOKENIZER = "flores200"
FP16 = False

RESULTS_DIR = Path("/content/final_stronger_bt_experiment")
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
EXAMPLES_DIR = RESULTS_DIR / "examples"
SCORES_DIR = RESULTS_DIR / "scores"
for d in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR, EXAMPLES_DIR, SCORES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Previous valid Colab folder, if still present in the runtime.
PREVIOUS_RESULTS_DIR = Path("/content/mt5_bn_backtranslation_results")

config = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "base_pool_size_for_same_split": BASE_POOL_SIZE,
    "synthetic_candidates": SYNTHETIC_CANDIDATES,
    "synthetic_target": SYNTHETIC_TARGET,
    "min_filtered_synthetic": MIN_FILTERED_SYNTHETIC,
    "baseline_epochs_if_needed": BASELINE_EPOCHS_IF_NEEDED,
    "reverse_epochs": REVERSE_EPOCHS,
    "improved_epochs": IMPROVED_EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adafactor",
    "fp16": FP16,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
    "bleu_tokenizer": BLEU_TOKENIZER,
    "gpu": torch.cuda.get_device_name(0),
}
pd.DataFrame([{"Parameter": k, "Value": v} for k, v in config.items()]).to_csv(
    TABLES_DIR / "training_configuration.csv", index=False
)
print(json.dumps(config, indent=2, ensure_ascii=False))

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    text = str(value).replace("\u200c", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def word_count(text):
    return len(str(text).split())


def get_full_dataset_examples():
    url = f"https://datasets-server.huggingface.co/info?dataset={DATASET_NAME}&config={DATASET_CONFIG}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return int(r.json()["dataset_info"]["splits"]["train"]["num_examples"])
    except Exception as exc:
        print("Metadata lookup failed:", exc)
        return None


def stream_clean_pairs(total_unique):
    dataset = load_dataset("parquet", data_files=DATASET_PARQUET, split="train", streaming=True)
    seen = set()
    pairs = []
    scanned = empty_removed = duplicate_removed = 0
    for row in dataset:
        scanned += 1
        en = clean_text(row.get("src"))
        bn = clean_text(row.get("tgt"))
        if not en or not bn:
            empty_removed += 1
            continue
        key = (en, bn)
        if key in seen:
            duplicate_removed += 1
            continue
        seen.add(key)
        pairs.append({"english": en, "bangla": bn})
        if len(pairs) >= total_unique:
            break
    return pairs, {
        "candidate_rows_scanned": scanned,
        "empty_rows_removed": empty_removed,
        "duplicate_pairs_removed": duplicate_removed,
        "clean_unique_pairs": len(pairs),
    }


full_examples = get_full_dataset_examples()
total_needed = BASE_POOL_SIZE + SYNTHETIC_CANDIDATES
pairs, preprocessing_stats = stream_clean_pairs(total_needed)
base_pairs = pairs[:BASE_POOL_SIZE]
extra_pairs = pairs[BASE_POOL_SIZE:]

shuffled = list(base_pairs)
random.Random(SEED).shuffle(shuffled)
train_pairs = shuffled[:TRAIN_SIZE]
val_pairs = shuffled[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
test_pairs = shuffled[TRAIN_SIZE + VAL_SIZE:TRAIN_SIZE + VAL_SIZE + TEST_SIZE]
base_mono = shuffled[TRAIN_SIZE + VAL_SIZE + TEST_SIZE:]

mono_pairs = (base_mono + extra_pairs)[:SYNTHETIC_CANDIDATES]
assert len(train_pairs) == TRAIN_SIZE
assert len(val_pairs) == VAL_SIZE
assert len(test_pairs) == TEST_SIZE
assert len(mono_pairs) >= SYNTHETIC_TARGET

dataset_stats = pd.DataFrame([
    {
        "Split": "Full Samanantar bn train metadata",
        "Sentence pairs": full_examples,
        "Avg English length (words)": "Not computed; full corpus not downloaded",
        "Avg Bangla length (words)": "Not computed; full corpus not downloaded",
    },
    {
        "Split": "Train",
        "Sentence pairs": len(train_pairs),
        "Avg English length (words)": round(sum(word_count(x["english"]) for x in train_pairs) / len(train_pairs), 2),
        "Avg Bangla length (words)": round(sum(word_count(x["bangla"]) for x in train_pairs) / len(train_pairs), 2),
    },
    {
        "Split": "Validation",
        "Sentence pairs": len(val_pairs),
        "Avg English length (words)": round(sum(word_count(x["english"]) for x in val_pairs) / len(val_pairs), 2),
        "Avg Bangla length (words)": round(sum(word_count(x["bangla"]) for x in val_pairs) / len(val_pairs), 2),
    },
    {
        "Split": "Test",
        "Sentence pairs": len(test_pairs),
        "Avg English length (words)": round(sum(word_count(x["english"]) for x in test_pairs) / len(test_pairs), 2),
        "Avg Bangla length (words)": round(sum(word_count(x["bangla"]) for x in test_pairs) / len(test_pairs), 2),
    },
    {
        "Split": "BT candidate monolingual pool",
        "Sentence pairs": len(mono_pairs),
        "Avg English length (words)": round(sum(word_count(x["english"]) for x in mono_pairs) / len(mono_pairs), 2),
        "Avg Bangla length (words)": round(sum(word_count(x["bangla"]) for x in mono_pairs) / len(mono_pairs), 2),
    },
])
dataset_stats.to_csv(TABLES_DIR / "dataset_statistics.csv", index=False)
pd.DataFrame(pairs).to_csv(RESULTS_DIR / "cleaned_candidate_pairs.csv", index=False)
display(dataset_stats)
print("Preprocessing:", preprocessing_stats)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)


def to_dataset(rows):
    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)


def preprocess_dataset(rows, direction):
    ds = to_dataset(rows)

    def preprocess(batch):
        if direction == "en_bn":
            src = [f"translate English to Bengali: {x}" for x in batch["english"]]
            tgt = batch["bangla"]
        elif direction == "bn_en":
            src = [f"translate Bengali to English: {x}" for x in batch["bangla"]]
            tgt = batch["english"]
        else:
            raise ValueError(direction)
        model_inputs = tokenizer(src, max_length=MAX_SOURCE_LENGTH, truncation=True)
        labels = tokenizer(text_target=tgt, max_length=MAX_TARGET_LENGTH, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return ds.map(preprocess, batched=True, remove_columns=ds.column_names, desc=f"Tokenizing {direction}")


tokenized_train_en_bn = preprocess_dataset(train_pairs, "en_bn")
tokenized_val_en_bn = preprocess_dataset(val_pairs, "en_bn")
tokenized_train_bn_en = preprocess_dataset(train_pairs, "bn_en")
tokenized_val_bn_en = preprocess_dataset(val_pairs, "bn_en")

In [ ]:
def training_args(output_dir, epochs):
    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": epochs,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "adafactor": True,
        "max_grad_norm": 1.0,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "save_strategy": "no",
        "report_to": "none",
        "predict_with_generate": False,
        "fp16": FP16,
        "dataloader_num_workers": 2,
        "remove_unused_columns": True,
    }
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    kwargs["eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"] = "epoch"
    return Seq2SeqTrainingArguments(**kwargs)


def train_model(run_name, train_ds, val_ds, epochs):
    started = time.perf_counter()
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.config.decoder_start_token_id = tokenizer.pad_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args(RESULTS_DIR / "checkpoints" / run_name, epochs),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    train_output = trainer.train()
    eval_metrics = trainer.evaluate()
    elapsed = time.perf_counter() - started
    if not math.isfinite(float(train_output.training_loss)) or float(train_output.training_loss) <= 0:
        raise RuntimeError(f"Invalid train loss for {run_name}: {train_output.training_loss}")
    if not math.isfinite(float(eval_metrics["eval_loss"])) or float(eval_metrics["eval_loss"]) <= 0:
        raise RuntimeError(f"Invalid eval loss for {run_name}: {eval_metrics['eval_loss']}")
    hist = pd.DataFrame(trainer.state.log_history)
    hist["run_name"] = run_name
    hist.to_csv(SCORES_DIR / f"{run_name}_trainer_log_history.csv", index=False)
    summary = {
        "run_name": run_name,
        "epochs": epochs,
        "train_runtime_seconds": round(elapsed, 2),
        "train_loss": float(train_output.training_loss),
        "validation_loss": float(eval_metrics["eval_loss"]),
    }
    print(summary)
    return model, trainer, hist, summary


BAD_WORDS_IDS = [
    tokenizer.encode(f"<extra_id_{i}>", add_special_tokens=False)
    for i in range(100)
    if tokenizer.encode(f"<extra_id_{i}>", add_special_tokens=False)
]


def generate_texts(model, texts, source_language):
    model.eval()
    prefix = "translate English to Bengali: " if source_language == "english" else "translate Bengali to English: "
    preds = []
    started = time.perf_counter()
    with torch.no_grad():
        for start in range(0, len(texts), GENERATION_BATCH_SIZE):
            batch = [prefix + x for x in texts[start:start + GENERATION_BATCH_SIZE]]
            enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH).to(model.device)
            gen = model.generate(
                **enc,
                max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                num_beams=4,
                do_sample=False,
                no_repeat_ngram_size=3,
                bad_words_ids=BAD_WORDS_IDS,
            )
            preds.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return [clean_text(x) for x in preds], time.perf_counter() - started


def nonempty_rate(preds):
    return sum(bool(str(x).strip()) for x in preds) / max(1, len(preds))


def corpus_bleu(preds, refs):
    score = sacrebleu.corpus_bleu(preds, [refs], tokenize=BLEU_TOKENIZER)
    return {"bleu": float(score.score), "signature": score.format(signature=True), "tokenizer": BLEU_TOKENIZER}


def cleanup(*objs):
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Baseline: reuse previous valid baseline if present; otherwise train it.
test_sources = [x["english"] for x in test_pairs]
test_refs = [x["bangla"] for x in test_pairs]
previous_manifest_path = PREVIOUS_RESULTS_DIR / "manifest.json"

baseline_model = baseline_trainer = None
if previous_manifest_path.exists():
    baseline_source_note = "Reused previous valid Colab baseline metrics from /content/mt5_bn_backtranslation_results."
    previous_manifest = json.loads(previous_manifest_path.read_text(encoding="utf-8"))
    baseline_bleu = previous_manifest["baseline_bleu"]
    baseline_summary = previous_manifest["baseline_summary"]
    previous_examples = pd.read_csv(PREVIOUS_RESULTS_DIR / "tables" / "sample_translation_comparison.csv")
    baseline_predictions_for_examples = previous_examples["Baseline Prediction"].head(10).fillna("").tolist()
    baseline_source_for_examples = previous_examples["Source English"].head(10).tolist()
    baseline_ref_for_examples = previous_examples["Reference Bangla"].head(10).tolist()
    baseline_history = pd.read_csv(PREVIOUS_RESULTS_DIR / "scores" / "baseline_en_bn_trainer_log_history.csv")
    print("Reused previous valid baseline metrics from", PREVIOUS_RESULTS_DIR)
else:
    baseline_source_note = "Baseline was retrained in this stronger notebook because no previous valid baseline folder was found."
    baseline_model, baseline_trainer, baseline_history, baseline_summary = train_model(
        "baseline_en_bn", tokenized_train_en_bn, tokenized_val_en_bn, BASELINE_EPOCHS_IF_NEEDED
    )
    baseline_predictions, baseline_generation_seconds = generate_texts(baseline_model, test_sources, "english")
    if nonempty_rate(baseline_predictions) < 0.8:
        raise RuntimeError("Baseline predictions mostly empty; invalid run.")
    baseline_bleu = corpus_bleu(baseline_predictions, test_refs)
    baseline_predictions_for_examples = baseline_predictions[:10]
    baseline_source_for_examples = test_sources[:10]
    baseline_ref_for_examples = test_refs[:10]

print("Baseline BLEU:", baseline_bleu)
print("Baseline summary:", baseline_summary)

In [ ]:
# Train stronger reverse model and generate synthetic candidates.
reverse_model, reverse_trainer, reverse_history, reverse_summary = train_model(
    "reverse_bn_en_for_filtered_backtranslation",
    tokenized_train_bn_en,
    tokenized_val_bn_en,
    REVERSE_EPOCHS,
)
mono_bangla = [x["bangla"] for x in mono_pairs]
synthetic_english_raw, synthetic_generation_seconds = generate_texts(reverse_model, mono_bangla, "bangla")
print("Raw synthetic non-empty rate:", nonempty_rate(synthetic_english_raw))

raw_synth = pd.DataFrame({"synthetic_english": synthetic_english_raw, "bangla": mono_bangla})
raw_synth.to_csv(EXAMPLES_DIR / "synthetic_backtranslation_raw_candidates.csv", index=False)
cleanup(reverse_model, reverse_trainer)

In [ ]:
def ascii_letter_ratio(text):
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 0.0
    letters = sum(c.isascii() and c.isalpha() for c in chars)
    return letters / len(chars)


def punctuation_ratio(text):
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 1.0
    punct = sum((not c.isalnum()) for c in chars)
    return punct / len(chars)


def max_token_repetition_ratio(text):
    toks = text.lower().split()
    if not toks:
        return 1.0
    counts = Counter(toks)
    return max(counts.values()) / len(toks)


def is_good_synthetic(en, bn):
    en = clean_text(en)
    if not en:
        return False, "empty"
    toks = en.split()
    if len(toks) < 3:
        return False, "too_short"
    if len(en) < 10:
        return False, "too_short_chars"
    if punctuation_ratio(en) > 0.45:
        return False, "mostly_punctuation"
    if ascii_letter_ratio(en) < 0.55:
        return False, "not_english_like"
    if max_token_repetition_ratio(en) > 0.45 and len(toks) >= 5:
        return False, "repeated_generic"
    bn_len = max(1, word_count(bn))
    ratio = len(toks) / bn_len
    if ratio < 0.25 or ratio > 4.0:
        return False, "bad_length_ratio"
    bad_exact = {"the", "and", "but", "however", "therefore", "i don't know"}
    if en.lower().strip(" .,!?:;\"'") in bad_exact:
        return False, "generic_exact"
    return True, "kept"


filtered = []
filter_counts = Counter()
seen_en = set()
seen_pair = set()
for en, bn in zip(synthetic_english_raw, mono_bangla):
    keep, reason = is_good_synthetic(en, bn)
    if not keep:
        filter_counts[reason] += 1
        continue
    en_clean = clean_text(en)
    key_en = en_clean.lower()
    key_pair = (key_en, bn)
    if key_en in seen_en or key_pair in seen_pair:
        filter_counts["duplicate"] += 1
        continue
    seen_en.add(key_en)
    seen_pair.add(key_pair)
    filtered.append({
        "english": en_clean,
        "bangla": bn,
        "creation_method": "Filtered reverse mT5-small Bangla-to-English back-translation",
    })
    if len(filtered) >= SYNTHETIC_TARGET:
        break

filtered_synth = pd.DataFrame(filtered)
filtered_synth.to_csv(EXAMPLES_DIR / "synthetic_backtranslation_pairs_filtered.csv", index=False)
pd.DataFrame([{"reason": k, "count": v} for k, v in filter_counts.items()]).to_csv(
    TABLES_DIR / "synthetic_filtering_summary.csv", index=False
)
print("Filtered synthetic pairs:", len(filtered_synth))
print("Filter counts:", filter_counts)
display(filtered_synth.head(10))
if len(filtered_synth) < MIN_FILTERED_SYNTHETIC:
    raise RuntimeError(f"Only {len(filtered_synth)} filtered synthetic pairs; required at least {MIN_FILTERED_SYNTHETIC}.")

In [ ]:
# Train improved model on original + filtered synthetic pairs.
improved_train_pairs = train_pairs + filtered_synth[["english", "bangla"]].to_dict("records")
tokenized_improved_train = preprocess_dataset(improved_train_pairs, "en_bn")

improved_model, improved_trainer, improved_history, improved_summary = train_model(
    "improved_en_bn_filtered_synthetic",
    tokenized_improved_train,
    tokenized_val_en_bn,
    IMPROVED_EPOCHS,
)
improved_predictions, improved_generation_seconds = generate_texts(improved_model, test_sources, "english")
if nonempty_rate(improved_predictions) < 0.8:
    raise RuntimeError("Improved predictions mostly empty; invalid run.")
improved_bleu = corpus_bleu(improved_predictions, test_refs)
print("Improved BLEU:", improved_bleu)
print("Improved summary:", improved_summary)

In [ ]:
# Build example comparison and automatic Better/Worse notes.
if previous_manifest_path.exists():
    example_sources = baseline_source_for_examples
    example_refs = baseline_ref_for_examples
    source_to_improved = dict(zip(test_sources, improved_predictions))
    improved_for_examples = [source_to_improved.get(src, "") for src in example_sources]
else:
    example_sources = test_sources[:10]
    example_refs = test_refs[:10]
    improved_for_examples = improved_predictions[:10]


def sent_bleu(pred, ref):
    try:
        return float(sacrebleu.sentence_bleu(str(pred), [str(ref)], tokenize=BLEU_TOKENIZER).score)
    except Exception:
        return 0.0


notes = []
for base_pred, imp_pred, ref in zip(baseline_predictions_for_examples, improved_for_examples, example_refs):
    b = sent_bleu(base_pred, ref)
    i = sent_bleu(imp_pred, ref)
    if i > b + 0.01:
        label = f"Better (sentence BLEU {i:.2f} vs {b:.2f})"
    elif i < b - 0.01:
        label = f"Worse (sentence BLEU {i:.2f} vs {b:.2f})"
    else:
        label = f"No clear change (sentence BLEU {i:.2f} vs {b:.2f})"
    notes.append(label)

examples = pd.DataFrame({
    "Source English": example_sources,
    "Reference Bangla": example_refs,
    "Baseline Prediction": baseline_predictions_for_examples,
    "Improved Prediction": improved_for_examples,
    "Better / Worse note": notes,
})
examples.to_csv(TABLES_DIR / "translation_examples.csv", index=False)
display(examples)

In [ ]:
# Save metrics, markdown files, and charts.
baseline_bleu_value = float(baseline_bleu["bleu"])
improved_bleu_value = float(improved_bleu["bleu"])
bleu_delta = improved_bleu_value - baseline_bleu_value
bleu_pct = (bleu_delta / baseline_bleu_value * 100) if baseline_bleu_value else float("nan")
baseline_val = float(baseline_summary["validation_loss"])
improved_val = float(improved_summary["validation_loss"])
val_delta = improved_val - baseline_val

final_dataset_stats = pd.concat([
    dataset_stats,
    pd.DataFrame([
        {
            "Split": "Raw synthetic BT candidates",
            "Sentence pairs": len(synthetic_english_raw),
            "Avg English length (words)": round(sum(word_count(x) for x in synthetic_english_raw) / max(1, len(synthetic_english_raw)), 2),
            "Avg Bangla length (words)": round(sum(word_count(x) for x in mono_bangla) / max(1, len(mono_bangla)), 2),
        },
        {
            "Split": "Filtered synthetic BT pairs used",
            "Sentence pairs": len(filtered_synth),
            "Avg English length (words)": round(sum(word_count(x) for x in filtered_synth["english"]) / max(1, len(filtered_synth)), 2),
            "Avg Bangla length (words)": round(sum(word_count(x) for x in filtered_synth["bangla"]) / max(1, len(filtered_synth)), 2),
        },
        {
            "Split": "Improved training total",
            "Sentence pairs": len(improved_train_pairs),
            "Avg English length (words)": round(sum(word_count(x["english"]) for x in improved_train_pairs) / max(1, len(improved_train_pairs)), 2),
            "Avg Bangla length (words)": round(sum(word_count(x["bangla"]) for x in improved_train_pairs) / max(1, len(improved_train_pairs)), 2),
        },
    ])
], ignore_index=True)
final_dataset_stats.to_csv(TABLES_DIR / "dataset_statistics_final.csv", index=False)

bleu_df = pd.DataFrame([
    {"Model": "Baseline mT5-small", "BLEU": round(baseline_bleu_value, 4), "Signature": baseline_bleu["signature"]},
    {"Model": "Improved filtered BT mT5-small", "BLEU": round(improved_bleu_value, 4), "Signature": improved_bleu["signature"]},
])
loss_df = pd.DataFrame([
    {"Model": "Baseline mT5-small", "Validation loss": round(baseline_val, 4), "Training loss": round(float(baseline_summary["train_loss"]), 4), "Training time seconds": baseline_summary["train_runtime_seconds"]},
    {"Model": "Reverse mT5-small", "Validation loss": round(float(reverse_summary["validation_loss"]), 4), "Training loss": round(float(reverse_summary["train_loss"]), 4), "Training time seconds": reverse_summary["train_runtime_seconds"]},
    {"Model": "Improved filtered BT mT5-small", "Validation loss": round(improved_val, 4), "Training loss": round(float(improved_summary["train_loss"]), 4), "Training time seconds": improved_summary["train_runtime_seconds"]},
])
bleu_df.to_csv(TABLES_DIR / "bleu_comparison.csv", index=False)
loss_df.to_csv(TABLES_DIR / "validation_loss_comparison.csv", index=False)

all_history = pd.concat([
    baseline_history.assign(run_name="baseline_en_bn") if "run_name" not in baseline_history.columns else baseline_history,
    reverse_history,
    improved_history,
], ignore_index=True, sort=False)
all_history.to_csv(SCORES_DIR / "training_log_history_all_models.csv", index=False)

def write_md(name, text):
    (RESULTS_DIR / name).write_text(text, encoding="utf-8")

write_md("dataset_statistics.md", final_dataset_stats.to_markdown(index=False))
training_config_md = pd.DataFrame([{"Parameter": k, "Value": v} for k, v in config.items()]).to_markdown(index=False)
write_md("training_configuration.md", training_config_md + "\n\nBaseline source: " + baseline_source_note + "\n")
write_md("bleu_comparison.md", bleu_df.to_markdown(index=False))
write_md("validation_loss_comparison.md", loss_df.to_markdown(index=False))
write_md("translation_examples.md", examples.to_markdown(index=False))

qual = f'''# Qualitative Analysis

The stronger filtered back-translation run generated {len(filtered_synth)} filtered synthetic pairs from {len(synthetic_english_raw)} raw candidates.

Baseline BLEU: {baseline_bleu_value:.4f}
Improved BLEU: {improved_bleu_value:.4f}
BLEU change: {bleu_delta:.4f} ({bleu_pct:.2f}%)

Review the example table for sentence-level Better/Worse notes. The notes are based on sentence BLEU against the reference Bangla and should be interpreted as automatic indicators, not human judgments.
'''
write_md("qualitative_analysis.md", qual)

discussion = f'''# Final Discussion Notes

This stronger test used the same 5,000/500/500 train/validation/test split as the previous valid Colab run. {baseline_source_note} It increased synthetic generation to {SYNTHETIC_CANDIDATES} candidates and retained {len(filtered_synth)} filtered synthetic pairs for improved-model training.

Baseline BLEU was {baseline_bleu_value:.4f}. Improved BLEU was {improved_bleu_value:.4f}. The BLEU change was {bleu_delta:.4f}, or {bleu_pct:.2f}%.

Baseline validation loss was {baseline_val:.4f}. Improved validation loss was {improved_val:.4f}. The validation-loss change was {val_delta:.4f}.

Conclusion: {'the stronger filtered back-translation setup improved BLEU.' if bleu_delta > 0 else 'the stronger filtered back-translation setup did not improve BLEU.'} If BLEU still drops, likely causes include noisy synthetic sources from the reverse model, domain mismatch in synthetic pairs, and insufficient reverse-model quality even after filtering.
'''
write_md("final_discussion_notes.md", discussion)

manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": config,
    "full_samanantar_bn_train_examples": full_examples,
    "preprocessing": preprocessing_stats,
    "raw_synthetic_candidates": len(synthetic_english_raw),
    "filtered_synthetic_pairs": len(filtered_synth),
    "filter_counts": dict(filter_counts),
    "baseline_source_note": baseline_source_note,
    "baseline_bleu": baseline_bleu,
    "improved_bleu": improved_bleu,
    "baseline_summary": baseline_summary,
    "reverse_summary": reverse_summary,
    "improved_summary": improved_summary,
}
(RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

plt.figure(figsize=(7,4))
plt.bar(["Baseline", "Improved"], [baseline_bleu_value, improved_bleu_value], color=["#34699a", "#c4552f"])
plt.ylabel("BLEU")
plt.title("BLEU Comparison: Stronger Filtered BT")
for idx, value in enumerate([baseline_bleu_value, improved_bleu_value]):
    plt.text(idx, value, f"{value:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bleu_comparison_chart.png", dpi=200)
plt.show()

plt.figure(figsize=(7,4))
plt.bar(["Baseline", "Improved"], [baseline_val, improved_val], color=["#34699a", "#c4552f"])
plt.ylabel("Validation loss")
plt.title("Validation Loss Comparison: Stronger Filtered BT")
for idx, value in enumerate([baseline_val, improved_val]):
    plt.text(idx, value, f"{value:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_loss_chart.png", dpi=200)
plt.show()

loss_logs = all_history[all_history["loss"].notna()].copy()
plt.figure(figsize=(9,5))
for run_name, group in loss_logs.groupby("run_name"):
    plt.plot(group["step"], group["loss"], label=run_name)
plt.xlabel("Optimizer step")
plt.ylabel("Training loss")
plt.title("Training Loss: Stronger Filtered BT")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_loss_chart.png", dpi=200)
plt.show()

zip_path = shutil.make_archive(str(RESULTS_DIR), "zip", root_dir=RESULTS_DIR)
print("Created:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Download unavailable:", exc)